# Exercise 10: Text mining applications

In exercises 6 to 9, you learned the concepts of text mining on four small documents. Now, let's use them on a **real corpus**: **4,000 movie reviews** written on the French website Allociné, each labelled **positif** (positive) or **négatif** (negative) according to the rating given by its author.

We go through four applications, from the simplest to the most advanced:

| Part | Application | Main tool | Supervised? |
|---|---|---|---|
| 10a | Word clouds and distinctive words | `CountVectorizer`, `wordcloud` | no |
| 10b | Sentiment analysis with a pre-trained model | `transformers` | no |
| 10c | Supervised classification | TF-IDF + `LogisticRegression` | **yes** |
| 10d | Semantic search | `sentence-transformers` | no |

The data are stored in `data/2_text_mining/exercice_10/` (see `codebook.md` for their origin):

| File | Rows | Balance |
|---|---|---|
| `critiques_films_train.csv` | 3,000 | 1,500 positive / 1,500 negative |
| `critiques_films_test.csv` | 1,000 | 500 positive / 500 negative |

⚠️ The reviews are **in French**: the models and the stopwords used in this notebook are chosen accordingly.

### How to work

- Run the cells **one by one, from top to bottom** (`Shift + Enter`).
- ✏️ marks a cell you have to complete: replace the lines marked `## @COMPLETE`.
- 🧪 marks a test cell: run it to check your answer before moving on.
- 💡 gives you a hint, 🤔 asks you a question to think about.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/2_text_mining/exercice_10")

## Load the reviews

Each file has two columns: `text` (the review) and `polarite` (`positif` or `négatif`). The **train** set is used to learn, the **test** set to evaluate.

In [ ]:
train = pd.read_csv(DATA_PATH / "critiques_films_train.csv")
test = pd.read_csv(DATA_PATH / "critiques_films_test.csv")

print("Train:", len(train), "reviews | Test:", len(test), "reviews")
train.head(3)

We also need a list of **French stopwords** (`scikit-learn` only provides English ones). We define a short one here. Note that it also contains `film` and `films`: in a corpus of movie reviews, these words are everywhere and tell us nothing.

In [ ]:
STOPWORDS_FR = set('''
au aux avec ce ces dans de des du elle en et eux il je la le les leur lui ma
mais me même mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se
ses son sur ta te tes toi ton tu un une vos votre vous c d j l à m n s t y été
être ai as avait avais avons ont est sont a als cette cet plus moins très bien
fait film films tout tous toute toutes comme aussi car donc alors
sans sous entre vers chez ni leurs cela ça là où dont quand
'''.split())

print(len(STOPWORDS_FR), "stopwords")

## Exercise 10a: Word clouds and distinctive words

*The simplest application: descriptive, without any model.*

**What for?** It is the first thing to do with an unknown corpus: see at a glance what it is about, and what **distinguishes two groups**. For instance: compare the vocabulary of two political parties in parliamentary debates, or summarise thousands of customer reviews.

**How?** We count the words of the positive reviews and of the negative reviews, as in the term-document matrix of exercise 6. Then, for each word, we compute its **log-odds**: it compares the frequency of the word in the positive reviews with its frequency in the negative ones.

$$\text{log-odds}(w) = \log\left(\frac{\text{frequency of } w \text{ in the positive reviews}}{\text{frequency of } w \text{ in the negative reviews}}\right)$$

- log-odds $> 0$: the word is more frequent in the positive reviews,
- log-odds $< 0$: the word is more frequent in the negative reviews,
- log-odds $\approx 0$: the word is used as much in both (like *acteur* or *histoire*).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Term-document matrix: one row per review, one column per word (present in at least 10 reviews)
vectorizer = CountVectorizer(min_df=10, stop_words=list(STOPWORDS_FR))
counts = vectorizer.fit_transform(train["text"])
words = np.array(vectorizer.get_feature_names_out())

# Total number of occurrences of each word in the positive and in the negative reviews
positive = np.asarray(counts[(train["polarite"] == "positif").values].sum(axis=0)).ravel()
negative = np.asarray(counts[(train["polarite"] == "négatif").values].sum(axis=0)).ravel()

# Smoothed log-odds: the +1 avoids dividing by zero for a word absent from one of the classes
log_odds = np.log((positive + 1) / (positive.sum() + 1)) - np.log((negative + 1) / (negative.sum() + 1))
order = log_odds.argsort()

print("Most POSITIVE words:", ", ".join(words[order[::-1][:15]]))
print("Most NEGATIVE words:", ", ".join(words[order[:15]]))

Now, visually, with one **word cloud** per polarity. A word cloud built on raw frequencies would be flooded with words used everywhere (*acteur*, *scène*, *histoire*...). So we size each word by its **distinctive power** (the absolute value of its log-odds), not by its frequency.

In [ ]:
from wordcloud import WordCloud

positive_weights = {word: score for word, score in zip(words, log_odds) if score > 0}
negative_weights = {word: -score for word, score in zip(words, log_odds) if score < 0}


def plot_wordcloud(weights, title, ax):
    """Draw a word cloud where the size of each word is given by its weight."""
    cloud = WordCloud(width=500, height=350, background_color="white", colormap="viridis")
    ax.imshow(cloud.generate_from_frequencies(weights))
    ax.axis("off")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_wordcloud(positive_weights, "Distinctive words of the positive reviews", axes[0])
plot_wordcloud(negative_weights, "Distinctive words of the negative reviews", axes[1])
plt.show()

✏️ **Your turn**: display the **30** most distinctive words of each polarity instead of 15. Are the new words still meaningful?

In [ ]:
## @COMPLETE : print the 30 most positive and the 30 most negative words

## Exercise 10b: Sentiment analysis with a pre-trained model

*One step further: we use a model already trained by someone else, as a black box.*

**What for?** Put a **measure** (positive / negative) on volumes of text that we could never read by hand: follow the tone of tweets about a reform over time, measure the satisfaction in open survey answers, ...

**How?** The `transformers` library (from Hugging Face) loads in one line a **multilingual** model that predicts a sentiment, **without any training on our side**. We apply it to 200 reviews of the test set and compare its predictions with the true labels.

<img src="img/sentiment.png" width="560" alt="Sentiment analysis">

⏳ The first run downloads the model (about 500 MB) and may take a few minutes.

In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
)

sample = test.sample(200, random_state=42).copy()

# truncation=True: the model reads at most about 512 tokens, the end of very long reviews is ignored
predictions = sentiment_model(sample["text"].tolist(), truncation=True, batch_size=16)

# The model answers in English, we translate its labels to compare them with ours
translation = {"positive": "positif", "negative": "négatif", "neutral": "neutre"}
sample["predicted"] = [translation[p["label"]] for p in predictions]

pd.crosstab(sample["polarite"], sample["predicted"], rownames=["true"], colnames=["predicted"])

The model has **never seen** our corpus, and already does well. Advantage: no training at all. Limit: we control neither what it has learned, nor its categories (it adds a `neutre` class that does not exist in our data).

✏️ **Your turn**: find the reviews where the model **disagrees** with the true label, count them, and read a few of them. Why is the model wrong?

💡 *Hint*: keep the rows of `sample` where the column `polarite` is different from the column `predicted`.

In [ ]:
## @COMPLETE : keep the rows where the model is wrong
errors = None

## @COMPLETE : print how many there are, and display a few of them

⚠️ A sentiment model has been trained on some data and may carry **biases**: it often misreads irony, negations, or praise specific to a domain (what is positive for a movie is not always positive elsewhere). Its output is a **measure with errors**, not the truth. Before relying on it in a study, **validate it** on a sample labelled by hand to estimate its error rate.

## Exercise 10c: Supervised classification

*This time, we train a model ourselves.*

**What for?** It is the heart of **automated coding**: with a few hundred texts labelled by hand, we teach the machine to label all the others (sort 200,000 comments, classify court decisions by type, ...), with a **reliability measure** (accuracy, confusion matrix).

**How?** We turn the reviews into **TF-IDF** vectors (exercise 7), then train a **logistic regression** to predict the polarity. The model learns on the **train** set, and we evaluate it on the **test** set, which it has never seen.

<img src="img/classification.png" width="560" alt="Supervised classification">

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

# TF-IDF vectors: the vocabulary and the IDF are learned on the train set only
tfidf = TfidfVectorizer(min_df=5, stop_words=list(STOPWORDS_FR))
X_train = tfidf.fit_transform(train["text"])
X_test = tfidf.transform(test["text"])

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, train["polarite"])

predicted = classifier.predict(X_test)
print("Accuracy on the test set:", round(accuracy_score(test["polarite"], predicted), 3))

ConfusionMatrixDisplay.from_predictions(test["polarite"], predicted, cmap="Blues")
plt.show()

A logistic regression is a **linear** model: it gives one weight to each word. We can read these weights to see the words that push the most towards *positif* or *négatif*. We find the same kind of words as in 10a, but this time they are **learned** to make the best predictions.

In [ ]:
vocabulary = np.array(tfidf.get_feature_names_out())
weights = classifier.coef_[0]  # classes are ['négatif', 'positif']: a positive weight pushes towards 'positif'
order = weights.argsort()

print("Towards POSITIVE:", ", ".join(vocabulary[order[::-1][:15]]))
print("Towards NEGATIVE:", ", ".join(vocabulary[order[:15]]))

✏️ **Your turn**: write your own review (in French) and let the model predict its polarity. Then try an **ironic** sentence, like *« génial, j'ai adoré m'ennuyer pendant deux heures »*: is the model fooled?

💡 *Hint*: a new text must go through the **same** TF-IDF transformation as the training texts: `classifier.predict(tfidf.transform(["your review"]))`.

In [ ]:
## @COMPLETE : predict the polarity of your own reviews

## Exercise 10d: Semantic search

*We go back to the dense embeddings of exercise 8 (SBERT).*

**What for?** Explore a large corpus **by meaning**, without guessing the exact words used: find all the passages of interviews that talk about job insecurity even when the word does not appear, build a thematic sub-corpus, ... It is also the basic building block of modern search engines.

**How?** We encode each review with a **multilingual** sentence model, then rank the reviews by **cosine similarity** with a query written in natural language, exactly like the `search` function of exercise 9.

<img src="img/search.png" width="560" alt="Semantic search">

⏳ Encoding the 1,000 reviews of the test set takes a minute or two.

In [ ]:
from sentence_transformers import SentenceTransformer, util

search_model = SentenceTransformer("distiluse-base-multilingual-cased-v2")
reviews = test["text"].tolist()
review_vectors = search_model.encode(reviews, batch_size=64)


def semantic_search(query, k=3):
    """Print the k reviews whose meaning is the closest to the query."""
    query_vector = search_model.encode([query])
    scores = util.cos_sim(query_vector, review_vectors).numpy().ravel()
    for i in scores.argsort()[::-1][:k]:
        print(f"[{scores[i]:.2f}] ({test['polarite'].iloc[i]}) {reviews[i][:200]}...\n")


semantic_search("un jeu d'acteur décevant et un scénario plat")

✏️ **Your turn**: search the reviews with your own query, for instance about the special effects, the soundtrack, or a movie for children. Do the results really talk about your topic, even when they don't use the same words?

💡 *Hint*: the model is multilingual: try the same query in English!

In [ ]:
## @COMPLETE : call semantic_search with your own query

## 🎉 Well done!

| Part | Application | Main tool | Supervised? |
|---|---|---|---|
| 10a | Word clouds / distinctive words | `CountVectorizer`, `wordcloud` | no |
| 10b | Sentiment analysis | `transformers` (pre-trained) | no |
| 10c | Classification | TF-IDF + `LogisticRegression` | **yes** |
| 10d | Semantic search | `sentence-transformers` | no |

From simple word counts to search by meaning, text mining offers a **toolbox** to explore a corpus.

⚠️ **One last reflex**: these tools are **not neutral**. A model inherits the biases of its training data, and our corpus itself is a particular sample (French reviews of one website). Documenting the corpus and testing the robustness of the results is part of the method.